# Решения: сравнение признаков

**Для преподавателя.** Эталон к `lesson.ipynb` и `homework.ipynb`. Не показывать ученикам до сдачи.

In [ ]:
from pathlib import Path
import pandas as pd


def find_trips_csv() -> Path:
    for p in (Path('trips.csv'), Path('../../data/trips.csv'), Path('../data/trips.csv')):
        if p.exists():
            return p.resolve()
    raise FileNotFoundError('trips.csv не найден')


TRIPS_PATH = find_trips_csv()
df = pd.read_csv(TRIPS_PATH)
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score


## Урок. 1–3. eval_feature

In [ ]:
def eval_feature(feature_name: str, random_state: int = 42):
    X = df[[feature_name]]
    y = df['duration_min']
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=random_state
    )
    pred = LinearRegression().fit(X_tr, y_tr).predict(X_te)
    return mean_squared_error(y_te, pred), r2_score(y_te, pred)


mse_d, r2_d = eval_feature('distance_km')
mse_h, r2_h = eval_feature('hour')
print('distance', round(mse_d, 2), round(r2_d, 3))
print('hour', round(mse_h, 2), round(r2_h, 3))

## Урок. 4. Таблица

In [ ]:
compare_table = pd.DataFrame([
    {'feature': 'distance_km', 'mse': mse_d, 'r2': r2_d},
    {'feature': 'hour', 'mse': mse_h, 'r2': r2_h},
])
print(compare_table)

## Урок. 5. BETTER_FEATURE

In [ ]:
BETTER_FEATURE = 'distance_km' if r2_d >= r2_h else 'hour'
print(BETTER_FEATURE)

## Урок. 6. WHY

In [ ]:
WHY = (
    f'По R² на test лучше {BETTER_FEATURE} '
    f'(distance R²={r2_d:.3f}, hour R²={r2_h:.3f}). '
    'Для отчёта фиксируем один признак и те же seed/test_size.'
)
print(WHY)

## Урок. 7. PROTOCOL

In [ ]:
PROTOCOL_NOTE = (
    'Разный seed меняет состав test — сравнение R² становится некорректным.'
)
print(PROTOCOL_NOTE)

## ДЗ. 1. Таблица

In [ ]:
rows = []
for f in ('distance_km', 'hour'):
    m, r = eval_feature(f)
    rows.append({'feature': f, 'mse': m, 'r2': r})
table = pd.DataFrame(rows)
print(table)

## ДЗ. 2. is_comfort

In [ ]:
df2 = df.copy()
df2['is_comfort'] = (df2['vehicle_type'] == 'comfort').astype(int)

def eval_feature_df(frame, feature_name: str, random_state: int = 42):
    X = frame[[feature_name]]
    y = frame['duration_min']
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=random_state
    )
    pred = LinearRegression().fit(X_tr, y_tr).predict(X_te)
    return mean_squared_error(y_te, pred), r2_score(y_te, pred)


mse_c, r2_c = eval_feature_df(df2, 'is_comfort')
beats_distance = r2_c > r2_d
print(r2_c, beats_distance)

## ДЗ. 3. REPORT_CHOICE

In [ ]:
REPORT_CHOICE = (
    f'В сдачу берём {BETTER_FEATURE}: '
    f'R² distance={r2_d:.3f}, R² hour={r2_h:.3f} на test (seed 42).'
)
print(REPORT_CHOICE)